In [3]:
import os
import json
import copy
import torch
import torchsparse
import scipy.sparse as sp
import numpy as np
import pickle as pkl
import pandas as pd
import random
from tqdm import tqdm
import torch_geometric.transforms as T
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer

In [3]:
torch.zeros(4205205, 2000)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [4]:
with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
    prop_typ_id_dict = pkl.load(f)

In [5]:
inputs = "titles_abstracts"
filename_pos_human = "../data/pubmed_articles_pos_human.csv"
filename_neg_human = "../data/pubmed_articles_neg_human.csv"

In [5]:
vectorizer = HashingVectorizer(ngram_range=(1, 4), stop_words="english", n_features=1000)

In [10]:
            
#             pos_data_human = pd.read_csv(filename_pos_human, chunksize=1000)
#             neg_data_human = pd.read_csv(filename_neg_human, chunksize=1000)
#             vec_fitted = False
#             file_counter = 0
#             for pos_chunk in tqdm(pos_data_human):
                
#                 pmids = pos_chunk['pmid']
                
#                 X_chunk = [f"{title} {abstract}" for title, abstract in zip(pos_chunk["title"], pos_chunk["abstract_plaintext"])]
                
#                 if not vec_fitted:
#                     X_trans = vectorizer.fit_transform(X_chunk)
#                     vec_fitted = True
#                 else:
#                     X_trans = vectorizer.transform(X_chunk)
                
#                 del X_chunk
                
#                 pmid_lst = []
#                 pmid2text_embedding = {}
#                 for pmid, X_trans_line in zip(pmids, X_trans):
#                     pmid_lst.append(pmid)
#                     pmid2text_embedding[pmid] = X_trans_line
#                 del X_trans_line
#                 del X_trans
                
               
                
#                 with open(f"../data/bow_embeddings_pmids/pmid2text_embedding_BoW{file_counter}.pkl", "wb") as f:
#                      pkl.dump({pmid: pmid2text_embedding[pmid] for pmid in pmid_lst}, f)
#                 file_counter+=1
                
#                 del pmid_lst
#                 del pmid2text_embedding
#             del pos_chunk
#             del pos_data_human
            


8585it [19:20,  7.40it/s]


In [11]:
#             for neg_chunk in tqdm(neg_data_human):
#                 pmids = neg_chunk['pmid']
                
#                 X_chunk = [f"{title} {abstract}" for title, abstract in zip(neg_chunk["title"], neg_chunk["abstract_plaintext"])]
#                 X_trans = vectorizer.fit_transform(X_chunk)
#                 del X_chunk
                
#                 pmid_lst = []
#                 pmid2text_embedding = {}
#                 for pmid, X_trans_line in zip(pmids, X_trans):
#                     pmid_lst.append(pmid)
#                     pmid2text_embedding[pmid] = X_trans_line
#                 del X_trans_line
#                 del X_trans
#                 with open(f"../data/bow_embeddings_pmids/pmid2text_embedding_BoW{file_counter}.pkl", "wb") as f:
#                      pkl.dump({pmid: pmid2text_embedding[pmid] for pmid in pmid_lst}, f)
#                 file_counter+=1
                
#                 del pmid_lst
#                 del pmid2text_embedding
#             del neg_chunk
#             del neg_data_human

5684it [14:06,  6.72it/s]


In [4]:
dir_path = "../data/bow_embeddings_pmids"

pmids_with_text = []

for file in tqdm(os.listdir(dir_path)):
    file_path = os.path.join(dir_path, file)
    with open(file_path, "rb") as f:
        pmid2text_embedding_tmp = pkl.load(f)
        for key, embedding in pmid2text_embedding_tmp.items():
            pmids_with_text.append(key)

100%|████████████████████████████████████| 14269/14269 [01:41<00:00, 140.26it/s]


In [5]:
len(pmids_with_text)

14268043

In [6]:
with open('../data/open_alex_graph.json') as f:
    oalex_graph_dict = json.load(f)

In [7]:
with open("../data/pmid_to_openAlex.json") as f:
    pmid_to_openAlex = json.load(f)

In [8]:
PMIDS = '../data/pmids.pkl'    
with open(PMIDS, 'rb') as f:
        pmids = pkl.load(f)

In [30]:
reserved_pmids = []
for ptype in pmids:

    for pos_neg in pmids[ptype]:
        for split in pmids[ptype][pos_neg]:
            reserved_pmids.extend(pmids[ptype][pos_neg][split])

In [31]:
reserved_pmids = list(set(reserved_pmids))

In [32]:
len(reserved_pmids)

118941

In [33]:
additional_pmids = random.sample(list(pmid_to_openAlex), k=1000000)

In [34]:
reserved_pmids.extend(additional_pmids)

In [35]:
reserved_pmids = list(set(reserved_pmids))
len(reserved_pmids)

1118941

In [36]:
pmid_to_openAlex_sample = {key: pmid_to_openAlex[str(key)] for key in reserved_pmids if str(key) in pmid_to_openAlex}

In [37]:
len(pmid_to_openAlex_sample)

1117568

In [39]:
# openAlex2pmid = {val: key for key, val in pmid_to_openAlex.items()}

In [40]:
openAlex2pmid_sample = {val: key for key, val in pmid_to_openAlex_sample.items()}

In [41]:
pmids_in_graph_sample = []

for key, val_list in tqdm(oalex_graph_dict.items()):
    if key in openAlex2pmid_sample:
        pmid_key = openAlex2pmid_sample[key]
        pmids_in_graph_sample.append(pmid_key)
        
    for val in val_list:
        if val in openAlex2pmid_sample:
            pmid_val = openAlex2pmid_sample[val]
            pmids_in_graph_sample.append(pmid_val)

100%|█████████████████████████████| 3090753/3090753 [00:11<00:00, 261039.08it/s]


In [42]:
len(pmids_in_graph_sample)

6030304

In [19]:
pmids_in_graph = []

for key, val_list in tqdm(oalex_graph_dict.items()):
    if key in openAlex2pmid:
        pmid_key = openAlex2pmid[key]
        pmids_in_graph.append(pmid_key)
        
    for val in val_list:
        if val in openAlex2pmid:
            pmid_val = openAlex2pmid[val]
            pmids_in_graph.append(pmid_val)

100%|█████████████████████████████| 3090753/3090753 [00:20<00:00, 151734.79it/s]


In [20]:
final_list_pmids = list(set(pmids_in_graph) & set(map(str, pmids_with_text)))

In [21]:
len(final_list_pmids)

3409752

In [43]:
final_list_pmids_sample = list(set(pmids_in_graph_sample) & set(pmids_with_text))

In [44]:
len(final_list_pmids_sample)

101223

In [22]:
srcs = list(set([openAlex2pmid[key] for key in oalex_graph_dict.keys()]) & set(final_list_pmids))

In [45]:
srcs_sample = list(set([openAlex2pmid_sample[key] for key in oalex_graph_dict.keys() if key in openAlex2pmid_sample]) \
                   & set(final_list_pmids_sample))

In [46]:
len(srcs_sample)

101222

In [47]:
final_list_pmids_sample = sorted(map(int, final_list_pmids_sample))

In [23]:
final_list_pmids = sorted(map(int, final_list_pmids))

In [48]:
len(final_list_pmids_sample)

101223

In [49]:
pmid2node_id_sample = {final_list_pmids_sample[i]:i for i in range(len(final_list_pmids_sample))}
src_lst_sample = []
dest_lst_sample = []
for pmid_key in tqdm(srcs_sample):
            src_node_id = pmid2node_id_sample[int(pmid_key)]
            key = pmid_to_openAlex_sample[pmid_key]
            vals = oalex_graph_dict[key]
            val_pmids = sorted([int(openAlex2pmid_sample[val]) for val in vals if val in openAlex2pmid_sample])
            for val_pmid in val_pmids:
                if val_pmid in pmid2node_id_sample:
                    dest_node_id = pmid2node_id_sample[int(val_pmid)]
                        
                    src_lst_sample.append(src_node_id)
                    dest_lst_sample.append(dest_node_id)

100%|███████████████████████████████| 101222/101222 [00:00<00:00, 203436.66it/s]


In [50]:
pmid2node_id_sample

{384: 0,
 861: 1,
 1469: 2,
 1858: 3,
 2092: 4,
 2404: 5,
 2975: 6,
 5889: 7,
 8070: 8,
 9352: 9,
 9376: 10,
 9954: 11,
 10295: 12,
 11208: 13,
 12805: 14,
 13072: 15,
 14155: 16,
 14362: 17,
 14548: 18,
 15665: 19,
 16701: 20,
 19028: 21,
 19348: 22,
 21786: 23,
 22945: 24,
 23136: 25,
 24330: 26,
 25411: 27,
 26261: 28,
 26697: 29,
 27766: 30,
 28152: 31,
 29755: 32,
 31088: 33,
 31123: 34,
 32907: 35,
 32961: 36,
 33616: 37,
 33710: 38,
 34127: 39,
 34144: 40,
 34151: 41,
 34782: 42,
 35003: 43,
 36459: 44,
 39112: 45,
 39134: 46,
 39895: 47,
 40713: 48,
 40795: 49,
 42207: 50,
 43035: 51,
 43106: 52,
 47348: 53,
 49229: 54,
 50017: 55,
 50295: 56,
 50442: 57,
 51284: 58,
 51587: 59,
 51839: 60,
 51964: 61,
 52001: 62,
 52157: 63,
 52188: 64,
 52767: 65,
 53722: 66,
 57959: 67,
 60260: 68,
 60434: 69,
 61720: 70,
 61994: 71,
 68132: 72,
 69724: 73,
 69727: 74,
 70653: 75,
 71139: 76,
 71267: 77,
 72162: 78,
 72787: 79,
 72830: 80,
 74618: 81,
 76016: 82,
 76069: 83,
 76841: 84,
 780

In [51]:
len(pmid2node_id_sample)

101223

In [52]:
len(src_lst_sample)

39051

In [53]:
len(dest_lst_sample)

39051

In [24]:
pmid2node_id = {final_list_pmids[i]:i for i in range(len(final_list_pmids))}
src_lst = []
dest_lst = []
for pmid_key in tqdm(srcs):
            src_node_id = pmid2node_id[int(pmid_key)]
            key = pmid_to_openAlex[pmid_key]
            vals = oalex_graph_dict[key]
            val_pmids = sorted([int(openAlex2pmid[val]) for val in vals if val in openAlex2pmid if val in openAlex2pmid])
            for val_pmid in val_pmids:
                if val_pmid in pmid2node_id:
                    dest_node_id = pmid2node_id[int(val_pmid)]
                        
                    src_lst.append(src_node_id)
                    dest_lst.append(dest_node_id)

100%|█████████████████████████████| 3090753/3090753 [00:28<00:00, 109680.83it/s]


In [25]:
pmid2node_id

{24: 0,
 26: 1,
 30: 2,
 39: 3,
 40: 4,
 42: 5,
 43: 6,
 45: 7,
 56: 8,
 58: 9,
 59: 10,
 63: 11,
 76: 12,
 79: 13,
 82: 14,
 85: 15,
 87: 16,
 90: 17,
 104: 18,
 123: 19,
 124: 20,
 155: 21,
 170: 22,
 250: 23,
 251: 24,
 298: 25,
 337: 26,
 361: 27,
 362: 28,
 364: 29,
 366: 30,
 368: 31,
 372: 32,
 373: 33,
 374: 34,
 377: 35,
 382: 36,
 384: 37,
 390: 38,
 409: 39,
 411: 40,
 426: 41,
 446: 42,
 459: 43,
 463: 44,
 466: 45,
 467: 46,
 472: 47,
 494: 48,
 535: 49,
 537: 50,
 543: 51,
 545: 52,
 653: 53,
 656: 54,
 665: 55,
 671: 56,
 706: 57,
 709: 58,
 710: 59,
 747: 60,
 796: 61,
 839: 62,
 861: 63,
 911: 64,
 914: 65,
 933: 66,
 937: 67,
 973: 68,
 981: 69,
 982: 70,
 996: 71,
 1003: 72,
 1043: 73,
 1052: 74,
 1076: 75,
 1080: 76,
 1084: 77,
 1089: 78,
 1093: 79,
 1132: 80,
 1133: 81,
 1141: 82,
 1143: 83,
 1145: 84,
 1155: 85,
 1156: 86,
 1158: 87,
 1170: 88,
 1213: 89,
 1243: 90,
 1246: 91,
 1247: 92,
 1248: 93,
 1259: 94,
 1260: 95,
 1269: 96,
 1273: 97,
 1291: 98,
 1333: 99,


In [26]:
len(pmid2node_id)

3409752

In [27]:
sorted(src_lst)

[3,
 3,
 3,
 3,
 3,
 3,
 4,
 4,
 4,
 4,
 4,
 4,
 5,
 5,
 5,
 5,
 9,
 9,
 10,
 12,
 12,
 13,
 13,
 13,
 13,
 13,
 13,
 13,
 13,
 16,
 16,
 21,
 21,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 28,
 31,
 31,
 36,
 40,
 40,
 46,
 46,
 51,
 51,
 51,
 56,
 59,
 67,
 67,
 67,
 67,
 71,
 71,
 72,
 77,
 79,
 80,
 96,
 96,
 97,
 104,
 105,
 105,
 107,
 112,
 136,
 140,
 140,
 141,
 141,
 145,
 145,
 145,
 159,
 159,
 159,
 160,
 160,
 160,
 160,
 160,
 160,
 166,
 167,
 167,
 168,
 169,
 176,
 176,
 181,
 181,
 182,
 187,
 187,
 187,
 190,
 190,
 202,
 204,
 204,
 211,
 211,
 211,
 211,
 211,
 216,
 224,
 225,
 225,
 226,
 227,
 228,
 231,
 234,
 241,
 246,
 247,
 251,
 255,
 264,
 265,
 267,
 276,
 276,
 277,
 278,
 284,
 284,
 286,
 286,
 286,
 288,
 295,
 297,
 297,
 297,
 297,
 297,
 304,
 308,
 314,
 315,
 315,
 316,
 319,
 322,
 322,
 322,
 326,
 326,
 326,
 329,
 329,
 335,
 353,
 353,
 353,
 362,
 362,
 362,
 362,
 365,
 368,
 368,
 381,
 381,
 381,
 381,
 382,
 383,
 383,
 383,
 383,
 383,
 384

In [28]:
sorted(dest_lst)

[2,
 2,
 2,
 2,
 2,
 4,
 4,
 4,
 4,
 4,
 4,
 6,
 6,
 6,
 7,
 7,
 7,
 7,
 7,
 8,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 10,
 10,
 10,
 11,
 11,
 11,
 12,
 12,
 12,
 12,
 13,
 13,
 13,
 13,
 14,
 14,
 14,
 15,
 15,
 16,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 19,
 19,
 19,
 19,
 19,
 20,
 20,
 21,
 21,
 21,
 27,
 28,
 28,
 28,
 28,
 29,
 29,
 29,
 30,
 30,
 30,
 30,
 30,
 30,
 31,
 31,
 32,
 32,
 32,
 32,
 34,
 36,
 36,
 37,
 38,
 38,
 38,
 38,
 38,
 38,
 39,
 39,
 40,
 40,
 40,
 42,
 42,
 42,
 42,
 42,
 42,
 42,
 42,
 42,
 42,
 43,
 43,
 43,
 43,
 44,
 46,
 46,
 46,
 46,
 46,
 47,
 47,
 47,
 47,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 48,
 50,
 51,
 52,
 52,
 52,
 52,
 56,
 57,
 57,
 57,
 57,
 57,
 57,
 57,
 57,
 57,
 57,
 57,
 57,
 57,
 59,
 59,
 59,
 59,
 59,
 59,
 59,
 60,
 60,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 64,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 65,
 6

In [29]:
len(src_lst)

16512720

In [30]:
del oalex_graph_dict

In [31]:
edge_index = torch.tensor([src_lst, dest_lst])

In [32]:
# with open(f"../data/edge_index.pkl", "wb") as f:
#     pkl.dump(edge_index, f)

In [54]:
edge_index_sample = torch.tensor([src_lst_sample, dest_lst_sample])

In [55]:
# with open(f"../data/edge_index_sample.pkl", "wb") as f:
#     pkl.dump(edge_index_sample, f)

In [56]:
# with open("../data/pmid2node_id_sample.json", "w") as f:
#     json.dump(pmid2node_id_sample, f)

In [33]:
# with open("../data/pmid2node_id.json", "w") as f:
#     json.dump(pmid2node_id, f)

In [34]:
# with open("../data/pmid_to_openAlex.json", "w") as f:
#      json.dump(pmid_to_openAlex, f)

In [35]:
del edge_index
del pmid_to_openAlex

In [2]:
# with open("../data/pmid_to_openAlex.json") as f:
#     pmid_to_openAlex = json.load(f)

In [3]:
# with open(f"../data/edge_index.pkl", "rb") as f:
#     edge_index = pkl.load(f)

In [ ]:
# with open("../data/pmid_to_openAlex.json", "r") as f:
#     pmid_to_openAlex = json.load(f)

In [ ]:
# pmid2node_id = {}
# for pmid, open_alex_id in tqdm(pmid_to_openAlex.items()):
#     if open_alex_id in open_alex2node_id:
#         pmid2node_id[pmid] = open_alex2node_id[open_alex_id]

In [ ]:
# with open("../data/pmid2node_id.json", "w") as f:
#     json.dump(pmid2node_id, f)

In [34]:
# with open("../data/pmid2node_id_sample.json", "w") as f:
#     json.dump(pmid2node_id_sample, f)

In [2]:
with open("../data/pmid2node_id.json", "r") as f:
    pmid2node_id = json.load(f)

In [3]:
len(pmid2node_id)

3409752

In [4]:
pmid2node_id = {int(key): value for key, value in tqdm(pmid2node_id.items())}

100%|████████████████████████████| 3409752/3409752 [00:00<00:00, 4294421.68it/s]


In [5]:
dir_path = "../data/bow_embeddings_pmids"

x = [0 for _ in range(len(pmid2node_id))]
entered = []

for file in tqdm(os.listdir(dir_path)):
    file_path = os.path.join(dir_path, file)
    with open(file_path, "rb") as f:
        pmid2text_embedding_tmp = pkl.load(f)
        for key, embedding in pmid2text_embedding_tmp.items():
            if key in pmid2node_id:
                entered.append(1)
                node_id = pmid2node_id[key]
                x[node_id] = embedding
                del embedding
        del pmid2text_embedding_tmp

100%|████████████████████████████████████| 14269/14269 [01:38<00:00, 144.45it/s]


In [6]:
x

[<1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 197 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 441 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 130 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 411 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 332 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 404 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 323 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 378 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'

In [7]:
len(entered)

4205205

In [8]:
del entered

In [9]:
len([el for el in x if type(el)==int])

0

In [10]:
del pmid2node_id

In [57]:
dir_path = "../data/bow_embeddings_pmids"

x_sample = [0 for _ in range(len(pmid2node_id_sample))]
entered = []

for file in tqdm(os.listdir(dir_path)):
    file_path = os.path.join(dir_path, file)
    with open(file_path, "rb") as f:
        pmid2text_embedding_tmp = pkl.load(f)
        for key, embedding in pmid2text_embedding_tmp.items():
            if key in pmid2node_id_sample:
                entered.append(1)
                node_id = pmid2node_id_sample[key]
                x_sample[node_id] = embedding

100%|████████████████████████████████████| 14269/14269 [01:22<00:00, 173.40it/s]


In [58]:
x_sample

[<1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 440 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 183 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 282 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 140 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 364 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 413 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 332 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'
 	with 259 stored elements in Compressed Sparse Row format>,
 <1x1000 sparse matrix of type '<class 'numpy.float64'>'

In [59]:
with open("../data/x_array_sample.pkl", "wb") as f:
    pkl.dump(x_sample, f)

In [60]:
x_tensor_sp_sample = sp.vstack(x_sample)

In [2]:
# with open("../data/x_array_sample.pkl", "rb") as f:
#     x_sample = pkl.load(f)

In [61]:
with open("../data/x_sparse.pkl", "wb") as f:
    pkl.dump(x_tensor_sp_sample, f)

<3116652x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 1072433889 stored elements in Compressed Sparse Row format>

In [62]:
x_tensor_sample = x_tensor_sp_sample.todense()

In [ ]:
torch.zeros(3116652, 5000)

In [63]:
with open("../data/x_tensor_sample.pkl", "wb") as f:
    pkl.dump(x_tensor_sample, f)

In [13]:
x_tensor

matrix([[0],
        [0],
        [0],
        ...,
        [0],
        [0],
        [0]])

In [40]:
x_sample_mat_sp = sp.vstack(x_sample)
# x_sample_mat_sp = x_sample_mat_sp.tocoo()

In [41]:
x_sample_tensor = x_sample_mat_sp.todense()

In [42]:
# with open("../data/x_tensor_sample.pkl", "wb") as f:
#     pkl.dump(x_sample_tensor, f)

In [21]:
pmids

{'is_rct': {'pos': {'train': [322634,
    30311848,
    6180562,
    9783191,
    26932695,
    19072879,
    27097736,
    15030839,
    352505,
    2006357,
    15761783,
    8712390,
    18626487,
    8384658,
    17106274,
    8034071,
    22700504,
    11784454,
    1621239,
    21561700,
    2759066,
    12488379,
    1989806,
    25339737,
    28280109,
    11298850,
    12351537,
    30621479,
    15129751,
    9250117,
    23657721,
    27285856,
    17494789,
    3957710,
    25815420,
    8655904,
    15976241,
    31505127,
    1867231,
    9422557,
    11078487,
    27755017,
    22226929,
    1897510,
    30397707,
    20538134,
    9303287,
    12868026,
    19559433,
    28490464,
    6704275,
    22407658,
    29108721,
    11927703,
    15046691,
    23820061,
    19342294,
    17140533,
    31228412,
    20931294,
    11069320,
    27813240,
    4085127,
    2497545,
    2803624,
    26429297,
    8971618,
    23193211,
    20466334,
    16139696,
    28606184,
    1

In [ ]:
edge_index

In [ ]:
data.edge_index

In [ ]:
data.train_mask

In [ ]:
data.val_mask

In [ ]:
data.test_mask

In [ ]:
data.y

In [ ]:
data